# ReAct RAG (with multiple tools)

In [ ]:
from rich import print
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
# groq_api_key=os.getenv("GROQ_API_KEY")
# llm=ChatGroq(groq_api_key=groq_api_key,model="meta-llama/llama-4-scout-17b-16e-instruct")

In [ ]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

api_wrapper_arxiv= ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

In [ ]:
# arxiv.invoke("Attension is all you need")

In [ ]:
from langchain_community.tools import  WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=500)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

In [ ]:
wiki.invoke("RAG defination")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

In [ ]:
tavily.invoke("Provide me the recent AI news?")

In [ ]:
tools = [arxiv, wiki, tavily]

In [ ]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="qwen/qwen3-32b")

In [ ]:
llm_with_tools=llm.bind_tools(tools=tools)

In [ ]:
llm_with_tools.invoke("Provide me the recent AI news?")

### Workflow

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage],add_messages]

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

In [ ]:
workflow = StateGraph(State)

workflow.add_node("tool_calling_llm", tool_calling_llm)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "tool_calling_llm")

workflow.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END
    }
)

workflow.add_edge("tools", "tool_calling_llm")


In [ ]:
react_app = workflow.compile()

In [ ]:
react_app

In [ ]:
messages=react_app.invoke(State(messages="When is varanasi movie is releasing"))
for m in messages["messages"]:
    m.pretty_print()

---

In [ ]:
import os
from langsmith import Client

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

print(prompt)

In [ ]:
print(client.pull_prompt("hwchase17/react",dangerously_pull_public_prompt=True))
print(client.pull_prompt("rlm/rag-prompt",dangerously_pull_public_prompt=True))

In [ ]:
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant"
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

In [ ]:
if __name__ == "__main__":
    queries = [
        "What is RAG (Retrieval Augmented Generation)?",
        "What are the recent papers on attention mechanisms?",
        "What are the latest AI news today?",
    ]

    for q in queries:
        print(f"\n[bold cyan]Query:[/bold cyan] {q}")
        result = agent_executor.invoke({"input": q})
        print(f"\n[bold green]Answer:[/bold green] {result['output']}\n")
        print("─" * 60)